In [4]:
from fastapi import FastAPI
import requests as r
import os
import json
from dotenv import load_dotenv
import redis
import json
import logging

ENV_PATH = "../.env"
load_dotenv(dotenv_path=ENV_PATH)

redis_client = redis.Redis(
    host=os.getenv("REDIS_HOST"),
    port=os.getenv("REDIS_PORT"),
    password=os.getenv("REDIS_PWD"),
    username=os.getenv("REDIS_USR"),
    decode_responses=True
)

NEOWS_URL_BASE = "https://api.nasa.gov/neo/rest/v1/"
FEED_ADDON = "feed?"
LOOKUP_ADDON = "neo/"
BROWSE_ADDON = "browse"
NEOWS_API_KEY = os.getenv("NASA_API_KEY", "DEMO_KEY")
API_KEY_ADDON = f"api_key=xT5YNApZWjXKWov6NTJB4RWjzuh9jN4LLILliC1l"
CACHE_TTL = 86400 # impostato ad 1 giorno per developing

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

#caching helpers
""" formato key:
        feed: f_YYYY-MM-DD_YYYY-MM-DD, geenrata solo se le date sono impostate
        lookup: l_0000000
        browse: b, browse non cambia spesso e non devono essere salvate caratteristiche della query, 
                "b" sarà l'unica chiave necessaria
"""

def get_from_cache(key: str) -> dict | None:
    value = redis_client.get(key)
    if value is None:
        return None
    logger.info("Cache HIT per key '%s'", key)
    return json.loads(value)

def set_in_cache(key: str, data: dict) -> None:
    redis_client.setex(key, CACHE_TTL, json.dumps(data))
    logger.info("Cache SET per key '%s'", key)

#-----------------------------------------------------------

def neows_feed(start_date:str = None, end_date:str = None):
    """ formato data YYYY-MM-DD per start_date e end_date """

    url = NEOWS_URL_BASE + FEED_ADDON
    if (start_date and end_date): # se l'utente ha inserito date di ricerca
        url += f"start_date={start_date}&end_date={end_date}"
        key = f"f_{start_date}_{end_date}"
    
    #  se l'utente non ha inserito alcuna data di ricerca, il ? dopo feed nella costante FEED_ADDON
    #  non causa problematiche nella chiamata API   

    url += "&"+API_KEY_ADDON

    cache_response = get_from_cache(key)
    if cache_response: return cache_response
    
    # key non è presente nella cache
    print("cache failed")
    data_feed = json.loads(r.get(url).content)
    set_in_cache(key, data_feed)

    return data_feed
    

def neows_lookup(asteroid_id:int = None):
    if not asteroid_id: return {"error":"INSERT ASTEROID_ID"}
    key = f"l_{asteroid_id}"
    url = NEOWS_URL_BASE + LOOKUP_ADDON + str(asteroid_id) + "?" + API_KEY_ADDON

    cache_response = get_from_cache(key)
    if cache_response: return cache_response

    data_lookup = json.loads(r.get(url).content)
    set_in_cache(key, data_lookup)
    
    return data_lookup

def neows_browse():
    url = NEOWS_URL_BASE + LOOKUP_ADDON + BROWSE_ADDON + "?" + API_KEY_ADDON
    key = "b"

    cache_response = get_from_cache(key)
    if cache_response: return cache_response

    data_browse = json.loads(r.get(url).content)
    set_in_cache(key, data_browse)

    return data_browse

In [7]:
neows_feed("2026-03-02","2026-03-07")

cache failed


2026-05-02 19:16:35,961 | INFO | Cache SET per key 'f_2026-03-02_2026-03-07'


{'links': {'next': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2026-03-07&end_date=2026-03-12&detailed=false&api_key=xT5YNApZWjXKWov6NTJB4RWjzuh9jN4LLILliC1l',
  'previous': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2026-02-25&end_date=2026-03-02&detailed=false&api_key=xT5YNApZWjXKWov6NTJB4RWjzuh9jN4LLILliC1l',
  'self': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2026-03-02&end_date=2026-03-07&detailed=false&api_key=xT5YNApZWjXKWov6NTJB4RWjzuh9jN4LLILliC1l'},
 'element_count': 36,
 'near_earth_objects': {'2026-03-07': [{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/3182823?api_key=xT5YNApZWjXKWov6NTJB4RWjzuh9jN4LLILliC1l'},
    'id': '3182823',
    'neo_reference_id': '3182823',
    'name': '(2004 KG1)',
    'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=3182823',
    'absolute_magnitude_h': 23.96,
    'estimated_diameter': {'kilometers': {'estimated_diameter_min': 0.0429096504,
      'estimated_diameter_max': 0.0959488953},
     'met

In [6]:
neows_feed("2026-03-01","2026-03-07")

2026-05-02 19:14:33,468 | INFO | Cache HIT per key 'f_2026-03-01_2026-03-07'


{'links': {'next': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2026-03-07&end_date=2026-03-13&detailed=false&api_key=xT5YNApZWjXKWov6NTJB4RWjzuh9jN4LLILliC1l',
  'previous': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2026-02-23&end_date=2026-03-01&detailed=false&api_key=xT5YNApZWjXKWov6NTJB4RWjzuh9jN4LLILliC1l',
  'self': 'http://api.nasa.gov/neo/rest/v1/feed?start_date=2026-03-01&end_date=2026-03-07&detailed=false&api_key=xT5YNApZWjXKWov6NTJB4RWjzuh9jN4LLILliC1l'},
 'element_count': 39,
 'near_earth_objects': {'2026-03-07': [{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/3182823?api_key=xT5YNApZWjXKWov6NTJB4RWjzuh9jN4LLILliC1l'},
    'id': '3182823',
    'neo_reference_id': '3182823',
    'name': '(2004 KG1)',
    'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=3182823',
    'absolute_magnitude_h': 23.96,
    'estimated_diameter': {'kilometers': {'estimated_diameter_min': 0.0429096504,
      'estimated_diameter_max': 0.0959488953},
     'met

In [ ]:
print(redis_client.keys("*"))